In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_validate
from sklearn.metrics import recall_score, accuracy_score, f1_score, confusion_matrix


np.random.seed(42)
n_samples = 3000

data = {
    'claim_amount': np.random.lognormal(mean=8, sigma=1, size=n_samples),
    'past_claims': np.random.randint(0, 5, n_samples),
    'agent_tenure': np.random.randint(1, 20, n_samples),
    'policy_age': np.random.randint(1, 15, n_samples),
    'is_emergency': np.random.randint(0, 2, n_samples)
}
df = pd.DataFrame(data)
df['fraud'] = ((df['claim_amount'] > 5000) & (df['is_emergency'] == 1) & (df['past_claims'] > 2)).astype(int)

X = df.drop('fraud', axis=1)
y = df['fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1. Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
print("Top 3 Fraud Indicators (DT Rules):")
print(export_text(dt, feature_names=list(X.columns), max_depth=2))

# 2. Random Forest
rf_params = {'n_estimators': [50, 100, 200], 'max_depth': [10, 20, None], 'min_samples_split': [2, 5]}
rf_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_params, scoring='recall', cv=5)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

def calculate_custom_cost(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    fp = cm[0, 1]
    fn = cm[1, 0]
    return (fp * 1) + (fn * 10)

rf_preds = best_rf.predict(X_test)
dt_preds = dt.predict(X_test)

print(f"\nRF Total Cost: {calculate_custom_cost(y_test, rf_preds)}")
print(f"DT Total Cost: {calculate_custom_cost(y_test, dt_preds)}")


def compare_models(X, y, models_dict):
    results = []
    for name, model in models_dict.items():
        cv_results = cross_validate(model, X, y, cv=5, scoring=['accuracy', 'f1'], return_train_score=False)
        results.append({
            'Model': name,
            'Mean Acc': cv_results['test_accuracy'].mean(),
            'Std Acc': cv_results['test_accuracy'].std(),
            'Mean F1': cv_results['test_f1'].mean(),
            'Mean Fit Time': cv_results['fit_time'].mean()
        })
    return pd.DataFrame(results)


models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100)
}
print("\nModel Benchmarking:")
print(compare_models(X_train, y_train, models))

Top 3 Fraud Indicators (DT Rules):
|--- claim_amount <= 5130.35
|   |--- class: 0
|--- claim_amount >  5130.35
|   |--- past_claims <= 2.50
|   |   |--- class: 0
|   |--- past_claims >  2.50
|   |   |--- is_emergency <= 0.50
|   |   |   |--- class: 0
|   |   |--- is_emergency >  0.50
|   |   |   |--- class: 1


RF Total Cost: 0
DT Total Cost: 0

Model Benchmarking:
           Model  Mean Acc   Std Acc   Mean F1  Mean Fit Time
0  Decision Tree  0.999583  0.000833  0.996226       0.003582
1  Random Forest  1.000000  0.000000  1.000000       0.193921
